In [1]:
!pip install pypdf --upgrade --force-reinstall

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 329.6/329.6 kB 24.0 MB/s eta 0:00:00


In [3]:
!pip install openai yfinance duckduckgo-search langchain langchain-community langchain-huggingface sentence-transformers chromadb -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 67.3/67.3 kB 6.3 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 101.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.7/21.7 MB 115.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 24.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 78.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 63.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 484.9/484.9 kB 30.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.4/17.4 MB 135.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.5/72.5 kB 7.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 132.6/132.6 kB 13.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.4/66.4 kB 5.6 MB/s eta 0:

In [4]:
import warnings
import os
import sys
import time
import re
import requests
import pandas as pd
import yfinance as yf
from openai import OpenAI
from duckduckgo_search import DDGS
from google.colab import drive, userdata

# LangChain Kütüphaneleri
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import Chroma

# Gürültü Engelleyiciler
warnings.filterwarnings("ignore")
os.environ["PYTHONWARNINGS"] = "ignore"

# =============================================================================
# 1. BAĞLANTILAR
# =============================================================================
try:
    drive.mount('/content/drive')
except:
    pass

client = None
try:
    api_key = userdata.get('deep1') # Senin DeepSeek anahtarın
    if api_key is None: raise ValueError()

    client = OpenAI(
        api_key=api_key,
        base_url="https://api.deepseek.com"
    )
    print("✅ DeepSeek API Bağlantısı Başarılı!")
except:
    sys.exit("❌ HATA: Colab sol menüdeki Anahtar kısmına 'deep1' eklemelisin.")

# =============================================================================
# 2. RAG / PDF SİSTEMİ
# =============================================================================
if not os.path.exists("/content/kasko_policesi.pdf"):
    from langchain.docstore.document import Document
    docs = [Document(page_content="Kasko sigortası; çarpma, çarpılma, yanma, dolu, sel ve deprem hasarlarını kapsar. İkame araç süresi 7 gündür. Uzay mekiği kazaları kapsam dışıdır.")]
else:
    loader = PyPDFLoader("/content/kasko_policesi.pdf")
    docs = loader.load()

text_splitter = RecursiveCharacterTextSplitter(chunk_size=800, chunk_overlap=150)
splits = text_splitter.split_documents(docs)
embedding_model = HuggingFaceEmbeddings(model_name="sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2")
vector_db = Chroma.from_documents(documents=splits, embedding=embedding_model, collection_name=f"db_{int(time.time())}")

# =============================================================================
# 3. ARAÇLAR (TOOLS) - Ground Truth İçin de Kullanılacak
# =============================================================================
def web_search_tool(query: str):
    # print(f"   (Web aranıyor: {query[:15]}...)", end=" ") # Sessiz mod
    try:
        with DDGS() as ddgs:
            results = list(ddgs.text(keywords=query, region='tr-tr', safesearch='off', max_results=1))
        if results:
            summary = "\n".join([f"- {r['body']}" for r in results])
            return f"BULUNAN VERİLER:\n{summary}"
        return "İnternette sonuç bulunamadı."
    except Exception as e:
        return f"Web hatası: {e}"

def get_weather_dynamic(city_name: str):
    try:
        geo = requests.get(f"https://geocoding-api.open-meteo.com/v1/search?name={city_name.strip()}&count=1&language=tr&format=json", timeout=5).json()
        if 'results' not in geo: return "Şehir bulunamadı."
        lat, lon = geo['results'][0]['latitude'], geo['results'][0]['longitude']
        w = requests.get(f"https://api.open-meteo.com/v1/forecast?latitude={lat}&longitude={lon}&current_weather=true", timeout=5).json()
        return f"{geo['results'][0]['name']} Sıcaklık: {w['current_weather']['temperature']}°C"
    except: return "Hava durumu hatası."

def get_finance_dynamic(ticker: str):
    try:
        df = yf.Ticker(ticker.strip().upper()).history(period="1d")
        if df.empty: return "Veri yok."
        return f"{ticker.upper()} Fiyat: {round(df['Close'].iloc[-1], 2)}"
    except: return "Finans hatası."

def rag_tool(query: str):
    try:
        res = vector_db.similarity_search(query, k=2)
        return "\n".join([d.page_content for d in res]) if res else "Bilgi yok."
    except: return "PDF hatası."

known_tools = {
    "weather": get_weather_dynamic,
    "finance": get_finance_dynamic,
    "policy": rag_tool,
    "web": web_search_tool
}

# =============================================================================
# 4. AGENT (ASİSTAN)
# =============================================================================
system_prompt = """
Sen Yardımcı Asistansın. Soruya uygun tek bir araç seç.
ARAÇLAR:
1. finance: Borsa/Döviz (Parametre: Ticker örn: BTC-USD, TRY=X)
2. weather: Hava Durumu (Parametre: Şehir)
3. policy: Sigorta/Kasko.
4. web: Genel bilgi/Haberler.

FORMAT: Action: arac_adi: parametre
"""

def manager_agent(user_input):
    messages = [{"role": "system", "content": system_prompt}, {"role": "user", "content": f"SORU: {user_input}"}]

    for _ in range(1):
        try:
            resp = client.chat.completions.create(model="deepseek-chat", messages=messages, stop=["Observation:"], temperature=0)
            result = resp.choices[0].message.content.strip()

            if "Action:" in result:
                parts = result.split("Action:")[-1].split(":", 1)
                tool, inp = parts[0].strip(), parts[1].strip()
                if tool in known_tools:
                    obs = known_tools[tool](inp)
                    final_res = client.chat.completions.create(
                        model="deepseek-chat",
                        messages=messages + [{"role": "assistant", "content": result},
                                             {"role": "user", "content": f"Observation: {obs}\nCevap ver."}]
                    )
                    return final_res.choices[0].message.content.strip()
            return result.replace("Answer:", "").strip()
        except: return "Hata"
    return "Cevap yok."

# =============================================================================
# 5. TEST SETİ (40 SORU)
# =============================================================================
optimized_benchmark_set = [
    # --- FİNANS ---
    {"cat": "Finance", "type": "Normal", "q": "Dolar kaç TL?", "tgt": "TRY=X"},
    {"cat": "Finance", "type": "Normal", "q": "Bitcoin fiyatı nedir?", "tgt": "BTC-USD"},
    {"cat": "Finance", "type": "Normal", "q": "Türk Hava Yolları hissesi kaç TL?", "tgt": "THYAO.IS"},
    {"cat": "Finance", "type": "Normal", "q": "Altın ons fiyatı ne kadar?", "tgt": "GC=F"},
    {"cat": "Finance", "type": "Normal", "q": "Apple hisse değeri?", "tgt": "AAPL"},
    {"cat": "Finance", "type": "Zor", "q": "Bitcoinnn kaç dolar?", "tgt": "BTC-USD"},
    {"cat": "Finance", "type": "Zor", "q": "Japon Yeni alış satış?", "tgt": "JPYTRY=X"},
    {"cat": "Finance", "type": "Zor", "q": "FROTO hissesi ne durumda?", "tgt": "FROTO.IS"},
    {"cat": "Finance", "type": "Halüsinasyon", "q": "Wakanda parası ne kadar?", "tgt": "YOK"},
    {"cat": "Finance", "type": "Halüsinasyon", "q": "Wayne Enterprises hissesi kaç dolar?", "tgt": "YOK"},

    # --- HAVA DURUMU ---
    {"cat": "Weather", "type": "Normal", "q": "İstanbul hava durumu", "tgt": "Istanbul"},
    {"cat": "Weather", "type": "Normal", "q": "Ankara'da yağmur var mı?", "tgt": "Ankara"},
    {"cat": "Weather", "type": "Normal", "q": "İzmir kaç derece?", "tgt": "Izmir"},
    {"cat": "Weather", "type": "Normal", "q": "Londra hava durumu", "tgt": "London"},
    {"cat": "Weather", "type": "Normal", "q": "Antalya sıcaklık", "tgt": "Antalya"},
    {"cat": "Weather", "type": "Zor", "q": "Istanbull hava durumu", "tgt": "Istanbul"},
    {"cat": "Weather", "type": "Zor", "q": "Bayburt'ta hava nasıl?", "tgt": "Bayburt"},
    {"cat": "Weather", "type": "Zor", "q": "Kapadokya hava durumu", "tgt": "Nevsehir"},
    {"cat": "Weather", "type": "Halüsinasyon", "q": "Mars hava durumu kaç derece?", "tgt": "YOK"},
    {"cat": "Weather", "type": "Halüsinasyon", "q": "Narnia ülkesinde hava nasıl?", "tgt": "YOK"},

    # --- RAG / PDF ---
    {"cat": "RAG", "type": "Normal", "q": "İkame araç süresi kaç gündür?", "tgt": "7 gün"},
    {"cat": "RAG", "type": "Normal", "q": "Kasko neleri kapsar?", "tgt": "Çarpma, yanma, sel, deprem"},
    {"cat": "RAG", "type": "Normal", "q": "Deprem hasarı ödenir mi?", "tgt": "Evet"},
    {"cat": "RAG", "type": "Normal", "q": "Sel hasarı teminata dahil mi?", "tgt": "Evet"},
    {"cat": "RAG", "type": "Normal", "q": "Yangın durumu kaskoya girer mi?", "tgt": "Evet"},
    {"cat": "RAG", "type": "Zor", "q": "Ikame arac suresi ne kadar?", "tgt": "7 gün"},
    {"cat": "RAG", "type": "Zor", "q": "Yedek araba kaç gün veriliyor?", "tgt": "7 gün"},
    {"cat": "RAG", "type": "Zor", "q": "Policede deprem varmi?", "tgt": "Evet"},
    {"cat": "RAG", "type": "Halüsinasyon", "q": "Lastik patlaması garantiye girer mi?", "tgt": "Hayır / Bilgi Yok"},
    {"cat": "RAG", "type": "Halüsinasyon", "q": "Radyo çalınırsa ödenir mi?", "tgt": "Hayır / Bilgi Yok"},

    # --- WEB SEARCH ---
    {"cat": "Web", "type": "Normal", "q": "2024 asgari ücret ne kadar?", "tgt": "17002"},
    {"cat": "Web", "type": "Normal", "q": "Fiat Egea fiyat listesi", "tgt": "Fiyat bilgisi"},
    {"cat": "Web", "type": "Normal", "q": "Türkiye'nin başkenti neresi?", "tgt": "Ankara"},
    {"cat": "Web", "type": "Normal", "q": "iPhone 15 fiyatı ne kadar?", "tgt": "Fiyat"},
    {"cat": "Web", "type": "Normal", "q": "Dolar neden yükseliyor?", "tgt": "Ekonomik sebepler"},
    {"cat": "Web", "type": "Zor", "q": "Asgarii ucret kac tl oldu?", "tgt": "17002"},
    {"cat": "Web", "type": "Zor", "q": "OpenAI CEO'su kimdir?", "tgt": "Sam Altman"},
    {"cat": "Web", "type": "Zor", "q": "Togg T10X menzili ne kadar?", "tgt": "523 km"},
    {"cat": "Web", "type": "Halüsinasyon", "q": "iPhone 35 fiyatı ne kadar?", "tgt": "Böyle bir model yok"},
    {"cat": "Web", "type": "Halüsinasyon", "q": "Türkiye'nin Mars valisi kim?", "tgt": "Böyle biri yok"}
]

# =============================================================================
# 6. YENİ HAKEM (GROUND TRUTH DESTEKLİ)
# =============================================================================
def multi_judge_evaluation_with_truth(query, response, expected_tgt, q_type, category):

    # 1. GROUND TRUTH (GERÇEK VERİ) ÇEKME
    real_data_context = ""

    if q_type != "Halüsinasyon": # Sadece gerçek sorularda veri çekelim
        try:
            if category == "Finance":
                # 'tgt' alanında ticker kodu var (Örn: BTC-USD)
                real_val = get_finance_dynamic(expected_tgt)
                if "Hata" not in real_val and "yok" not in real_val:
                    real_data_context = f"\n*** REFERANS GERÇEK VERİ (API'dan Alındı): {real_val} ***\n"

            elif category == "Weather":
                # 'tgt' alanında şehir adı var (Örn: Istanbul)
                real_val = get_weather_dynamic(expected_tgt)
                if "Hata" not in real_val:
                    real_data_context = f"\n*** REFERANS GERÇEK VERİ (API'dan Alındı): {real_val} ***\n"
        except:
            pass # API hatası olursa boş geç, normal değerlendirsin

    # 2. HAKEM PROMPT'U
    judge_prompt = f"""
    Sen Denetçisin. Aşağıdaki cevabı değerlendir.

    SORU: {query}
    KATEGORİ: {category} ({q_type})
    BEKLENEN BİLGİ TÜRÜ: {expected_tgt}
    {real_data_context}

    MODEL CEVABI: {response}

    KURALLAR:
    1. DOGRULUK: Eğer yukarıda 'REFERANS GERÇEK VERİ' varsa, modelin cevabını onunla kıyasla. Sayısal yakınlık (Örn: 87.5k ile 87.8k) kabul edilir. Eğer referans yoksa mantıksal doğruluğa bak.
    2. HALUSINASYON: Bilinmeyen/Olmayan şeylerde (Halüsinasyon tipi) model "Yok/Bilmiyorum" dediyse 10 puan ver. Uydurduysa 0 ver.
    3. EKSIKSIZLIK: Cevap tatmin edici mi?
    4. USLUP: Nazik mi?

    FORMAT:
    DOGRULUK: [Puan]
    HALUSINASYON: [Puan]
    EKSIKSIZLIK: [Puan]
    USLUP: [Puan]
    GEREKCE: [Tek cümle]
    """

    try:
        resp = client.chat.completions.create(model="deepseek-chat", messages=[{"role": "user", "content": judge_prompt}], temperature=0)
        c = resp.choices[0].message.content
        return {
            "Dogruluk": int(re.search(r"DOGRULUK:\s*(\d+)", c).group(1)),
            "Halusinasyon": int(re.search(r"HALUSINASYON:\s*(\d+)", c).group(1)),
            "Eksiksizlik": int(re.search(r"EKSIKSIZLIK:\s*(\d+)", c).group(1)),
            "Uslup": int(re.search(r"USLUP:\s*(\d+)", c).group(1)),
            "Gerekce": re.search(r"GEREKCE:\s*(.*)", c).group(1)
        }
    except: return {"Dogruluk":0, "Halusinasyon":0, "Eksiksizlik":0, "Uslup":0, "Gerekce":"Hata"}

# =============================================================================
# 7. ÇALIŞTIRMA
# =============================================================================
print("🚀 GROUND TRUTH DESTEKLİ BENCHMARK BAŞLIYOR...")
results = []

for i, item in enumerate(optimized_benchmark_set):
    print(f"[{i+1}/40] {item['cat']:<8} | {item['q'][:35]:<35}", end=" ")

    start = time.time()
    try:
        ans = manager_agent(item['q'])
    except: ans = "HATA"

    # Yeni Hakem Fonksiyonu (Kategori bilgisini de gönderiyoruz)
    scores = multi_judge_evaluation_with_truth(item['q'], ans, item['tgt'], item['type'], item['cat'])
    avg = (scores['Dogruluk'] + scores['Halusinasyon'] + scores['Eksiksizlik']) / 3

    icon = "✅" if avg >= 8 else "⚠️" if avg >= 5 else "❌"
    print(f"-> {icon} Ort: {avg:.1f} ({time.time()-start:.1f}s)")

    results.append({
        "No": i+1, "Kategori": item['cat'], "Tip": item['type'], "Soru": item['q'],
        "Cevap": ans[:200], "Doğruluk": scores['Dogruluk'], "Halüsinasyon": scores['Halusinasyon'],
        "GENEL": round(avg, 1), "Gerekçe": scores['Gerekce']
    })

df = pd.DataFrame(results)
df.to_excel("ground_truth_benchmark.xlsx", index=False)
print("\n🎉 TEST BİTTİ! Kaydedildi: ground_truth_benchmark.xlsx")

Mounted at /content/drive
✅ DeepSeek API Bağlantısı Başarılı!


modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packag

-> ✅ Ort: 10.0 (12.7s)
[40/40] Web      | Türkiye'nin Mars valisi kim?        

/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packag

-> ✅ Ort: 10.0 (10.5s)

🎉 TEST BİTTİ! Kaydedildi: ground_truth_benchmark.xlsx


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packag

In [11]:
import warnings
import os
import sys
import time
import re
import requests
import pandas as pd
import yfinance as yf
from openai import OpenAI
from duckduckgo_search import DDGS
from google.colab import drive, userdata
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import Chroma

# Ayarlar
warnings.filterwarnings("ignore")
os.environ["PYTHONWARNINGS"] = "ignore"

# --- 1. AJAN (LOG TUTMA ÖZELLİKLİ) ---
def manager_agent_with_logging(user_input):
    """
    Hem cevabı hem de düşünce sürecini (log) döndürür.
    Return: (final_answer, log_string)
    """
    messages = [{"role": "system", "content": system_prompt},
                {"role": "user", "content": f"SORU: {user_input}"}]

    log_buffer = [] # Düşünceleri burada biriktireceğiz
    log_buffer.append(f"--- YENİ SORU: {user_input} ---")

    for _ in range(1):
        try:
            # 1. Adım: Düşünme
            start_think = time.time()
            resp = client.chat.completions.create(model="deepseek-chat", messages=messages, stop=["Observation:"], temperature=0)
            result = resp.choices[0].message.content.strip()

            log_buffer.append(f"[DÜŞÜNCE]: {result}")

            if "Action:" in result:
                parts = result.split("Action:")[-1].split(":", 1)
                tool, inp = parts[0].strip(), parts[1].strip()

                log_buffer.append(f"[EYLEM]: Araç Seçildi -> {tool} | Parametre -> {inp}")

                if tool in known_tools:
                    # 2. Adım: Araç Çalıştırma
                    obs = known_tools[tool](inp)
                    log_buffer.append(f"[GÖZLEM/API]: {str(obs)[:300]}...") # Log şişmesin diye kısalttık

                    # 3. Adım: Nihai Cevap
                    final_res = client.chat.completions.create(
                        model="deepseek-chat",
                        messages=messages + [{"role": "assistant", "content": result},
                                             {"role": "user", "content": f"Observation: {obs}\nCevap ver."}]
                    )
                    final_ans = final_res.choices[0].message.content.strip()
                    log_buffer.append(f"[NİHAİ CEVAP]: {final_ans}")

                    return final_ans, "\n".join(log_buffer)

            # Araç kullanmadıysa
            log_buffer.append(f"[EYLEM]: Araç kullanılmadı, direkt cevap verildi.")
            log_buffer.append(f"[NİHAİ CEVAP]: {result}")
            return result.replace("Answer:", "").strip(), "\n".join(log_buffer)

        except Exception as e:
            err_msg = f"Hata oluştu: {e}"
            log_buffer.append(f"[HATA]: {err_msg}")
            return "Hata", "\n".join(log_buffer)

    return "Cevap yok", "\n".join(log_buffer)

# --- 2. HAKEM (GROUND TRUTH İLE) ---
# (Önceki fonksiyonu aynen kullanıyoruz, tekrar yazmaya gerek yok ama referans olsun diye çağıracağız)

# --- 3. TESTİ ÇALIŞTIRMA VE KAYDETME ---
def run_benchmark_with_logs():
    print("🚀 DETAYLI KAYITLI BENCHMARK BAŞLIYOR (Excel + Düşünce Günlüğü)...")
    results = []

    # Log dosyasını açıyoruz
    log_filename = "ajan_dusunce_gunlugu.txt"
    with open(log_filename, "w", encoding="utf-8") as f:
        f.write("=== DEEPSEEK AGENT DÜŞÜNCE GÜNLÜĞÜ ===\n\n")

    for i, item in enumerate(optimized_benchmark_set):
        print(f"[{i+1}/40] {item['cat']:<8} | {item['q'][:30]:<30}", end=" ")

        start = time.time()

        # 1. Agent'ı Çağır (Loglu versiyon)
        ans, thought_log = manager_agent_with_logging(item['q'])

        # 2. Logu dosyaya yaz
        with open(log_filename, "a", encoding="utf-8") as f:
            f.write(thought_log + "\n" + "="*50 + "\n\n")

        # 3. Ground Truth Verisi Çekme (Hakem İçin)
        real_data_ref = ""
        if item['type'] != "Halüsinasyon":
            try:
                if item['cat'] == "Finance":
                    real_val = get_finance_dynamic(item['tgt'])
                    if "Hata" not in real_val: real_data_ref = real_val
                elif item['cat'] == "Weather":
                    real_val = get_weather_dynamic(item['tgt'])
                    if "Hata" not in real_val: real_data_ref = real_val
            except: pass

        # 4. Hakem Değerlendirmesi (Yeni Reference Data ile)
        # Not: multi_judge_evaluation_with_truth fonksiyonu önceki hücreden geliyor olmalı.
        # Eğer hata alırsan o fonksiyonu da buraya eklemem gerekir.
        try:
            scores = multi_judge_evaluation_with_truth(item['q'], ans, item['tgt'], item['type'], item['cat']) # type: ignore
        except:
            # Fonksiyon tanımlı değilse basit fallback (önceki hücre çalışmadıysa)
            scores = {"Dogruluk": 0, "Halusinasyon": 0, "Eksiksizlik": 0, "Uslup": 0, "Gerekce": "Hakem Hatası"}

        avg = (scores['Dogruluk'] + scores['Halusinasyon'] + scores['Eksiksizlik']) / 3
        icon = "✅" if avg >= 8 else "⚠️"
        print(f"-> {icon} Ort: {avg:.1f}")

        results.append({
            "No": i+1, "Kategori": item['cat'], "Tip": item['type'], "Soru": item['q'],
            "Cevap": ans[:150], "Genel Puan": round(avg, 1), "Gerekçe": scores['Gerekce']
        })

    # Excel Kaydı
    df = pd.DataFrame(results)
    df.to_excel("benchmark_ve_dusunce_raporu.xlsx", index=False)

    print("\n" + "="*60)
    print("🏁 İŞLEM TAMAMLANDI!")
    print("📂 1. Excel Raporu: benchmark_ve_dusunce_raporu.xlsx")
    print("📂 2. Düşünce Dosyası: ajan_dusunce_gunlugu.txt (Bunu indirip okuyabilirsin)")
    print("="*60)

# Başlat
run_benchmark_with_logs()

/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packag

🚀 DETAYLI KAYITLI BENCHMARK BAŞLIYOR (Excel + Düşünce Günlüğü)...
[1/40] Finance  | Dolar kaç TL?                  -> ✅ Ort: 10.0
[2/40] Finance  | Bitcoin fiyatı nedir?          -> ✅ Ort: 10.0
[3/40] Finance  | Türk Hava Yolları hissesi kaç  -> ✅ Ort: 10.0
[4/40] Finance  | Altın ons fiyatı ne kadar?     -> ✅ Ort: 10.0
[5/40] Finance  | Apple hisse değeri?            -> ✅ Ort: 10.0
[6/40] Finance  | Bitcoinnn kaç dolar?           -> ✅ Ort: 10.0
[7/40] Finance  | Japon Yeni alış satış?         -> ⚠️ Ort: 0.0
[8/40] Finance  | FROTO hissesi ne durumda?      

ERROR:yfinance:$FROTO: possibly delisted; no price data found  (period=1d) (Yahoo error = "No data found, symbol may be delisted")


-> ⚠️ Ort: 0.0
[9/40] Finance  | Wakanda parası ne kadar?       

ERROR:yfinance:$WAKANDA: possibly delisted; no price data found  (period=1d) (Yahoo error = "No data found, symbol may be delisted")


...

/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packag

-> ⚠️ Ort: 0.0
[40/40] Web      | Türkiye'nin Mars valisi kim?   

/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packag

-> ✅ Ort: 10.0

🏁 İŞLEM TAMAMLANDI!
📂 1. Excel Raporu: benchmark_ve_dusunce_raporu.xlsx
📂 2. Düşünce Dosyası: ajan_dusunce_gunlugu.txt (Bunu indirip okuyabilirsin)


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packag